<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Logistic Regression Project

*Session 4 · Notebook 03.05 · Practical · Student version*

## About this notebook

This is a hands-on **project lab** for logistic regression. You met the model in the lecture (notebook 03.04, on loan data); here you run the whole workflow yourself on a different, messier dataset. The emphasis is on the things that make logistic regression valuable in risk work:

- **Handling real missing data and mixed feature types** before modelling.
- **Reading the predicted probabilities**, passenger by passenger, as well as the overall metrics.
- **Reading the model beyond accuracy**: confusion matrix, ROC curve and AUC, and the decision threshold.

**scikit-learn documentation for the model(s) used in this notebook:** [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

## About the exercises

Each task appears as two cells: a **question** (markdown, sometimes with a `> *Hint:*`) and an empty **`# Your turn`** cell for you to write your answer. Work top to bottom, since later tasks reuse variables you create earlier; your coach has the worked solutions.

## About the data

`train_titanic.csv` is the well-known **Titanic** dataset: 891 passengers, with a target **`Survived`** (1 = survived, 0 = did not; 342 survived out of 891, about 38 percent). Predictors include `Pclass` (ticket class), `Sex`, `Age`, `SibSp` (siblings/spouses aboard), `Parch` (parents/children aboard), `Fare`, and `Embarked` (port).

It has genuine data-quality issues to deal with: `Age` is missing for 177 passengers, `Cabin` is missing for most, and `Embarked` has 2 blanks. That makes it a realistic practice ground.

## Why this matters for risk analysis

Logistic regression is the workhorse of credit scorecards and many other risk models, and for one big reason: **it is interpretable**. Every coefficient converts into an **odds ratio**, a plain-language statement like "being in third class multiplies the odds of survival by 0.3", or in a credit setting "each extra missed payment multiplies the odds of default by 1.8". Regulators, auditors, and declined customers can all be given a reason.

The transferable skills here are exactly those of a scorecard build:

- **Prepare mixed, missing data cleanly** (impute, encode) inside a leakage-safe pipeline.
- **Turn coefficients into odds ratios** so the model tells a story, not just a number.
- **Choose the decision threshold on purpose**, trading false alarms against missed events according to their business cost, rather than blindly using 0.5.

## Index

1. [Setup](#setup)
2. [Get the data](#data)
3. [Exploratory data analysis](#eda)
4. [Prepare the data (missing values and feature choice)](#prep)
5. [Train / test split](#split)
6. [Build and fit a logistic regression pipeline](#model)
7. [Evaluate: confusion matrix, report, ROC and AUC](#eval)
8. [Look at individual predictions](#odds)
9. [Move the decision threshold](#threshold)
10. [Further practice](#further)
11. [Key takeaways](#takeaways)

<a id="setup"></a>
## 1. Setup

**Before you start (Databricks): install seaborn.** The plots in this notebook use the `seaborn` library, which is not installed on every Databricks compute. Run the next two cells once, **first**: the first installs the library, the second restarts Python so the library can be imported. The restart clears every variable, which is why these cells come before everything else.

In [ ]:
%pip install seaborn

In [ ]:
# Restart Python so the newly installed library can be imported
dbutils.library.restartPython()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, accuracy_score)

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")

<a id="data"></a>
## 2. Get the data

### Exercise 1: Load the data

Read `../../datasets/train_titanic.csv` into a DataFrame called `titanic`. Show `titanic.head()`.

In [ ]:
# Extracting the data: 
titanic = pd.read_csv("https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/train_titanic.csv")
# Your turn


### Exercise 2: Inspect the data and missing values

Call `titanic.info()`, print the survival balance, and print the count of missing values per column (`isna().sum()`). Which columns have serious missingness?

In [ ]:
# Your turn


<a id="eda"></a>
## 3. Exploratory data analysis

A couple of plots to see which passengers survived.

### Exercise 3: Survival by sex and by class

Make two countplots side by side (or one at a time): survival counts split by `Sex`, and survival counts split by `Pclass`. Who was more likely to survive?

> *Hint:* `sns.countplot(data=titanic, x="Sex", hue="Survived")`.

In [ ]:
# Your turn


### Exercise 4: Age distribution

Draw a histogram of `Age` (dropping missing values). What is the rough age profile of the passengers?

> *Hint:* `titanic["Age"].dropna().hist(bins=30)`.

In [ ]:
# Your turn


<a id="prep"></a>
## 4. Prepare the data (missing values and feature choice)

We keep a sensible set of predictors and drop columns that are mostly missing or that are identifiers with no predictive value.

- **Drop** `PassengerId`, `Name`, `Ticket` (identifiers/text) and `Cabin` (687 of 891 missing).
- **Numeric** predictors: `Age`, `Fare`, `SibSp`, `Parch`. `Age` needs imputing.
- **Categorical** predictors: `Sex`, `Embarked`, `Pclass`. `Embarked` needs imputing.

Rather than clean columns by hand, we will let a `ColumnTransformer` inside a pipeline do the imputing and encoding, so it is fitted on the training data only (no leakage).

### Exercise 5: Choose features and target

Build `X` with the predictor columns `["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]` and `y` as `Survived`. Define two lists, `num_feats` and `cat_feats`, naming the numeric and categorical predictors (treat `Pclass` as categorical).

In [ ]:
# Your turn


<a id="split"></a>
## 5. Train / test split

### Exercise 6: Stratified train/test split

Split into train/test with `test_size=0.30`, `random_state=RANDOM_STATE`, and `stratify=y` so both sets keep the survival rate.

In [ ]:
# Your turn


<a id="model"></a>
## 6. Build and fit a logistic regression pipeline

We assemble a `ColumnTransformer` that:

- imputes missing `Age`/`Fare` with the median and standardises the numeric features, and
- imputes missing `Embarked` with the most frequent value and one-hot encodes the categorical features (`drop="first"` to avoid the dummy trap).

Then we chain it with a `LogisticRegression` in one pipeline.

### Exercise 7: Build the preprocessing + model pipeline

Build the numeric pipeline (`SimpleImputer(strategy="median")` then `StandardScaler()`), the categorical pipeline (`SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(drop="first", handle_unknown="ignore")`), combine them in a `ColumnTransformer` called `prep`, and put `prep` + `LogisticRegression(max_iter=1000)` into a pipeline called `logreg`. Fit it on the training data.

In [ ]:
# Your turn


<a id="eval"></a>
## 7. Evaluate: confusion matrix, report, ROC and AUC

### Exercise 8: Confusion matrix and classification report

Predict on `X_test` (call it `pred`), then print the accuracy, the confusion matrix, and the classification report.

In [ ]:
# Your turn


### Exercise 9: ROC curve and AUC

Get the predicted probabilities of survival with `logreg.predict_proba(X_test)[:, 1]`, compute the AUC with `roc_auc_score`, and plot the ROC curve (`roc_curve`). Add the diagonal "random" reference line.

> *Hint:* AUC summarises the ROC curve in one number; 0.5 is random, 1.0 is perfect.

In [ ]:
# Your turn


<a id="odds"></a>
## 8. Look at individual predictions

The metrics summarise the whole test set. It also helps to look at **individual passengers**: who does the model think is most and least likely to survive, and does that make sense?

### Exercise 10: Most and least likely survivors

Using the `proba` from Exercise 9, build a table of the test passengers with `Sex`, `Pclass`, `Age`, the predicted probability of survival and the actual outcome (`y_test`). Show the five passengers with the **highest** probability and the five with the **lowest**. Does the pattern make sense?

> *Hint:* copy the columns from `X_test`, add the two new columns, then use `sort_values(...)` and `head(5)`.

In [ ]:
# Your turn


<a id="threshold"></a>
## 9. Move the decision threshold

By default the model predicts "survived" when the predicted probability exceeds 0.5. That cut-off is a choice, not a law. Raising or lowering it trades **precision** against **recall**. In a risk setting you would set it by the relative cost of a false alarm versus a missed event.

### Exercise 11: Compare thresholds

Using the `proba` from Exercise 9, relabel predictions at thresholds 0.5, 0.4, and 0.3. For each, print the recall and precision on the survived class (class 1). What happens as you lower the threshold?

> *Hint:* `from sklearn.metrics import recall_score, precision_score`; `(proba >= t).astype(int)`.

In [ ]:
# Your turn


<a id="further"></a>
## 10. Further practice

1. **Feature engineering.** Create `FamilySize = SibSp + Parch + 1` and an `IsAlone` flag, add them, and see if AUC improves.
2. **Regularisation.** Logistic regression has a `C` parameter (inverse regularisation strength). Grid-search it with `GridSearchCV` and `scoring="roc_auc"`.
3. **Compare a model.** Fit a `RandomForestClassifier` on the same pipeline preprocessing and compare AUC. Does the more flexible model beat the interpretable one here?

### Exercise 12 (stretch): add a FamilySize feature

Add `FamilySize = SibSp + Parch + 1` to `X_train` and `X_test`, include it in `num_feats`, refit the pipeline, and compare the new AUC to the old one.

> *Hint:* rebuild the `ColumnTransformer` and pipeline after adding the feature to `num_feats`.

In [ ]:
# Your turn


<a id="takeaways"></a>
## 11. Key takeaways

| Step | What you did | Why it matters |
|---|---|---|
| Clean in a pipeline | Impute + encode via `ColumnTransformer` | Handles missing/mixed data with no leakage from the test set |
| Evaluate broadly | Confusion matrix, report, ROC/AUC | Accuracy alone misses how well probabilities rank cases |
| Individual predictions | `predict_proba` + `sort_values` | Checks that the model's ranking makes sense case by case |
| Move the threshold | `predict_proba` + chosen cut-off | Set the false-alarm vs missed-event balance to the business cost |

**The one-line lesson:** logistic regression earns its place in risk work by being explainable. Prepare the data cleanly, read it through its probabilities and the ROC curve, and choose the threshold deliberately.

Great job. You have built and, crucially, explained a logistic regression model end to end.